# Perceptrón con Scikit-Learn — clasificación del dataset Iris

**Módulo 06 — Fundamentos de redes neuronales.** Recurso descargable de la clase (`OD_RN1_ESP_M02_S05`), ampliado con explicaciones, métricas y correcciones.

## Objetivo

Entrenar un **perceptrón simple** con `sklearn.linear_model.Perceptron` para distinguir dos especies de flor iris (*setosa* y *versicolor*) a partir de dos medidas: el largo del sépalo y el largo del pétalo.

## Relación con la teoría del módulo

| Concepto de las slides | Dónde aparece acá |
|---|---|
| Estructura del perceptrón: `z = Σ wᵢ·xᵢ + b` | pesos en `coef_`, sesgo en `intercept_` |
| Función de activación escalón | interna al estimador; la salida de `predict` es 0 o 1 |
| Regla delta `wᵢ ← wᵢ + η·(d − y)·xᵢ` | la implementa `fit`; `eta0` es la tasa de aprendizaje η |
| Frontera de decisión lineal | la recta que graficamos al final |
| Limitación: solo problemas linealmente separables | por eso elegimos setosa vs versicolor, y por eso falla el XOR |

Este notebook es el **complemento aplicado** de [`practica_redes_neuronales.ipynb`](practica_redes_neuronales.ipynb), donde el mismo algoritmo está implementado a mano, celda por celda. Acá se usa la librería; allá se ve qué hace la librería por dentro.

## Qué se cambió respecto del recurso original

- Se agregó texto explicativo en cada paso (el original era solo código).
- Se fijó `random_state` en el split y en el modelo para que los resultados sean **reproducibles**.
- Se corrigió el cálculo de la exactitud: el original hacía `y = perceptron.predict(X_test)`, **pisando** la variable `y` con las etiquetas reales, y luego `accuracy_score(y, y_test)` con los argumentos en orden invertido. Acá se usa `y_pred` y el orden correcto `accuracy_score(y_test, y_pred)`.
- Se agregaron matriz de confusión, reporte de clasificación, comparación con la compuerta AND y la demostración del XOR.

## 1. Librerías

`scikit-learn` aporta el dataset, el split, el modelo y las métricas; `pandas` para manipular la tabla y `matplotlib` para graficar.

In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Semilla unica para todo el notebook: garantiza que los resultados se repitan
SEMILLA = 42

## 2. El dataset Iris

Iris es el conjunto de datos clásico de clasificación (Fisher, 1936): 150 flores, 50 de cada una de tres especies (*setosa*, *versicolor*, *virginica*), con cuatro medidas en centímetros por flor.

`load_iris()` devuelve un objeto tipo diccionario. Las claves que interesan son:

- `data`: matriz 150×4 con las medidas.
- `target`: vector de 150 etiquetas (0, 1, 2).
- `feature_names`: nombres de las 4 columnas.
- `target_names`: nombres de las 3 especies.

In [2]:
iris = load_iris()

print("Claves disponibles:", list(iris.keys()))
print("Forma de data:     ", iris.data.shape)
print("Atributos:         ", iris.feature_names)
print("Clases:            ", list(iris.target_names))

Claves disponibles: ['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module']
Forma de data:      (150, 4)
Atributos:          ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Clases:             [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


> **Tip:** en Jupyter, `load_iris?` muestra la documentación completa de la función (el recurso original usaba esa celda). Con `??` se ve además el código fuente.

## 3. Del array al DataFrame

Trabajar con un `DataFrame` de pandas hace mucho más legible el filtrado y la selección de columnas que hacer indexado sobre el array de NumPy.

In [3]:
data_df = pd.DataFrame(iris.data, columns=iris.feature_names)
data_df["target"] = iris.target

print("Filas por clase (0=setosa, 1=versicolor, 2=virginica):")
print(data_df["target"].value_counts().sort_index())

data_df.head()

Filas por clase (0=setosa, 1=versicolor, 2=virginica):
target
0    50
1    50
2    50
Name: count, dtype: int64


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## 4. Reducir el problema: 2 atributos y 2 clases

Dos decisiones deliberadas, y las dos vienen de las **limitaciones del perceptrón** vistas en la teoría:

1. **Dos clases (setosa y versicolor).** Un perceptrón simple es un clasificador **binario**: su salida es 0 o 1. Dejamos afuera *virginica*.
2. **Dos atributos (largo de sépalo y largo de pétalo).** No es una limitación del algoritmo — el perceptrón maneja cualquier cantidad de entradas — sino una elección **didáctica**: con dos atributos podemos dibujar los datos en un plano y ver la frontera de decisión como una recta.

Además, setosa y versicolor son **linealmente separables** con estos atributos, que es la condición que el perceptrón necesita para converger. Con versicolor vs virginica no lo serían y el entrenamiento nunca terminaría de estabilizarse.

In [4]:
columnas = ["sepal length (cm)", "petal length (cm)", "target"]
data_df = data_df[columnas]

# Nos quedamos solo con las clases 0 (setosa) y 1 (versicolor)
data_df = data_df[data_df["target"].isin([0, 1])]

print("Forma resultante:", data_df.shape)
data_df.describe()

Forma resultante: (100, 3)


,sepal length (cm),petal length (cm),target
count,100.000000,100.000000,100.000000
mean,5.471000,2.861000,0.500000
std,0.641698,1.449549,0.502519
min,4.300000,1.000000,0.000000
25%,5.000000,1.500000,0.000000
50%,5.400000,2.450000,0.500000
75%,5.900000,4.325000,1.000000
max,7.000000,5.100000,1.000000


## 5. Visualizar antes de modelar

Este gráfico es el paso más importante del notebook: **antes** de entrenar nada, muestra a ojo si las dos clases se pueden separar con una recta. Si los puntos estuvieran mezclados, el perceptrón no sería la herramienta adecuada.

In [5]:
setosa_data = data_df[data_df["target"] == 0]
versicolor_data = data_df[data_df["target"] == 1]

plt.figure(figsize=(7, 5))
plt.scatter(setosa_data["sepal length (cm)"], setosa_data["petal length (cm)"],
            label="setosa (0)", color="red", s=50)
plt.scatter(versicolor_data["sepal length (cm)"], versicolor_data["petal length (cm)"],
            label="versicolor (1)", color="blue", s=50)

plt.title("Sepal length vs Petal length")
plt.xlabel("Sepal length (cm)")
plt.ylabel("Petal length (cm)")
plt.grid()
plt.legend()
plt.show()

/var/folders/2k/hxbm582j23l33g7ttrfr_wh00000gn/T/ipykernel_50335/3011389595.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Los dos grupos quedan claramente aparte: hay un hueco vertical entre ellos en el eje del pétalo. Cualquier recta que pase por ese hueco separa las clases, así que existen **infinitas** soluciones válidas. El perceptrón encuentra una de ellas — cuál depende de la inicialización y del orden en que ve los ejemplos.

## 6. Separar atributos (`X`) y etiquetas (`y`)

Convención de scikit-learn: `X` es la matriz de atributos (una fila por muestra) e `y` el vector de etiquetas.

In [6]:
X = data_df[["sepal length (cm)", "petal length (cm)"]].values
y = data_df["target"].values

print("X:", X.shape, "| y:", y.shape)
print("Primeras 3 filas de X:\n", X[:3])
print("Primeras 3 etiquetas:", y[:3])

X: (100, 2) | y: (100,)
Primeras 3 filas de X:
 [[5.1 1.4]
 [4.9 1.4]
 [4.7 1.3]]
Primeras 3 etiquetas: [0 0 0]


## 7. División en entrenamiento y prueba

Se entrena con una parte de los datos y se evalúa con otra que el modelo **nunca vio**. Si midiéramos la exactitud sobre los mismos datos de entrenamiento, no sabríamos si el modelo aprendió o simplemente memorizó.

Dos parámetros que el recurso original no fijaba y conviene fijar:

- `random_state=SEMILLA`: hace la división **reproducible**. Sin esto, cada ejecución da un split distinto y por lo tanto pesos y exactitud distintos.
- `stratify=y`: mantiene en train y en test la misma proporción de cada clase (50/50 acá). Con particiones chicas, sin estratificar podría tocar un test desbalanceado.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA, stratify=y
)

print("Entrenamiento:", X_train.shape[0], "muestras")
print("Prueba:       ", X_test.shape[0], "muestras")
print("Clases en test:", np.bincount(y_test))

Entrenamiento: 80 muestras
Prueba:        20 muestras
Clases en test: [10 10]


## 8. Entrenar el perceptrón

`Perceptron` implementa exactamente el algoritmo de las slides. Los parámetros que usamos:

| Parámetro | Significado |
|---|---|
| `max_iter=10` | máximo de **épocas** (pasadas completas sobre el set de entrenamiento) |
| `eta0=0.05` | tasa de aprendizaje **η**: cuánto se corrigen los pesos en cada error |
| `random_state` | fija el barajado de los datos entre épocas → resultado reproducible |

En cada muestra el modelo calcula `z = w₁·x₁ + w₂·x₂ + b`, aplica el escalón (`1` si `z > 0`, si no `0`) y, **si se equivocó**, ajusta los pesos con la regla delta `wᵢ ← wᵢ + η·(d − y)·xᵢ`. Si acierta, no toca nada. Eso es todo lo que hace `fit`.

In [8]:
perceptron = Perceptron(max_iter=10, eta0=0.05, random_state=SEMILLA)
perceptron.fit(X_train, y_train)

print("Epocas efectivamente ejecutadas:", perceptron.n_iter_)

Epocas efectivamente ejecutadas: 7


Si `n_iter_` es menor que `max_iter`, el entrenamiento **convergió antes de tiempo**: dejó de haber errores (o mejoras) y el algoritmo cortó solo. Es la misma condición de corte que en la implementación manual del otro notebook.

## 9. Los pesos aprendidos

El modelo entrenado se resume en tres números: `w₁`, `w₂` y el sesgo `b`.

In [9]:
pesos = perceptron.coef_[0]
sesgo = perceptron.intercept_[0]

print(f"w1 (sepal length) = {pesos[0]:.4f}")
print(f"w2 (petal length) = {pesos[1]:.4f}")
print(f"sesgo (b)         = {sesgo:.4f}")
print(f"\nRegla aprendida: predice 1 si  {pesos[0]:.3f}*x1 + {pesos[1]:.3f}*x2 + {sesgo:.3f} > 0")

w1 (sepal length) = -0.1400
w2 (petal length) = 0.3100
sesgo (b)         = -0.0500

Regla aprendida: predice 1 si  -0.140*x1 + 0.310*x2 + -0.050 > 0


**Cómo leerlos:** lo que decide es el **signo de `z`**, y cada peso aporta según su magnitud y su signo. En esta corrida `w₂` (pétalo) sale positivo y con la mayor magnitud, mientras que `w₁` (sépalo) sale negativo y más chico: el largo del pétalo es el atributo que domina la decisión, y empuja hacia la clase versicolor. Coincide con el gráfico, donde la separación entre clases es básicamente vertical (a lo largo del eje del pétalo).

> **Cuidado con interpretar magnitudes de pesos** cuando los atributos están en escalas distintas. Acá ambos están en centímetros y en rangos parecidos, así que la comparación es válida. Si un atributo fuera en metros y otro en milímetros, habría que estandarizar (`StandardScaler`) antes de comparar.

## 10. Predecir sobre el conjunto de prueba

In [10]:
# Ojo: guardar en 'y_pred', NO en 'y' (el recurso original pisaba las etiquetas reales)
y_pred = perceptron.predict(X_test)

comparacion = pd.DataFrame({"real": y_test, "predicho": y_pred})
comparacion["acierto"] = comparacion["real"] == comparacion["predicho"]
comparacion

,real,predicho,acierto
0,1,1,True
1,1,1,True
2,1,1,True
3,1,1,True
4,0,0,True
5,0,0,True
6,0,0,True
7,1,1,True
8,0,0,True
9,0,0,True


## 11. Evaluar el modelo

La **exactitud** (*accuracy*) es la proporción de predicciones correctas. La convención de scikit-learn es `accuracy_score(y_verdadero, y_predicho)` — en ese orden.

La **matriz de confusión** muestra el detalle: cuántos de cada clase real fueron a cada clase predicha. La diagonal son los aciertos.

In [11]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Exactitud del perceptron: {accuracy:.2f}")

print("\nMatriz de confusion (filas = real, columnas = predicho):")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["real setosa", "real versicolor"],
    columns=["pred setosa", "pred versicolor"],
))

print("\nReporte de clasificacion:")
print(classification_report(y_test, y_pred, target_names=["setosa", "versicolor"]))

Exactitud del perceptron: 1.00

Matriz de confusion (filas = real, columnas = predicho):
                 pred setosa  pred versicolor
real setosa               10                0
real versicolor            0               10

Reporte de clasificacion:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00        10

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



Una exactitud de 1.00 acá **no significa que el modelo sea excelente**: significa que el problema es fácil. Las clases son linealmente separables y el conjunto de prueba tiene apenas 20 muestras. Un resultado perfecto en un problema separable es lo esperable, no un logro.

## 12. Clasificar una flor nueva

`predict` espera una matriz 2D (una fila por muestra). Por eso el `reshape(1, -1)`: convierte el vector `[6.2, 4.3]` en una matriz de 1×2.

In [12]:
flor_nueva = np.array([6.2, 4.3]).reshape(1, -1)   # sepalo 6.2 cm, petalo 4.3 cm
prediccion = perceptron.predict(flor_nueva)[0]

print(f"Prediccion: {prediccion} -> {iris.target_names[prediccion]}")

# decision_function devuelve el valor de z ANTES del escalon.
# Su signo da la clase; su magnitud, cuan lejos esta el punto de la frontera.
print(f"z = {perceptron.decision_function(flor_nueva)[0]:.4f}")

Prediccion: 1 -> versicolor
z = 0.4150


## 13. La frontera de decisión

La frontera es el conjunto de puntos donde el perceptrón duda, es decir donde `z = 0`:

$$w_1 x_1 + w_2 x_2 + b = 0$$

Despejando `x₂` queda la ecuación de una recta, que es lo que graficamos:

$$x_2 = -\frac{w_1}{w_2}\,x_1 - \frac{b}{w_2}$$

Todo lo que cae de un lado se clasifica como setosa y lo del otro lado como versicolor.

In [13]:
plt.figure(figsize=(8, 6))

plt.scatter(setosa_data["sepal length (cm)"], setosa_data["petal length (cm)"],
            label="setosa (0)", color="green", s=50)
plt.scatter(versicolor_data["sepal length (cm)"], versicolor_data["petal length (cm)"],
            label="versicolor (1)", color="blue", s=50)

# Resaltar las muestras del conjunto de prueba
plt.scatter(X_test[:, 0], X_test[:, 1], facecolors="none", edgecolors="black",
            s=160, linewidths=1.5, label="conjunto de prueba")

# Recta de separacion: x2 = -(w1/w2)*x1 - b/w2
x_values = np.linspace(4, 7.5, 100)
y_values = (-pesos[0] / pesos[1]) * x_values - sesgo / pesos[1]
plt.plot(x_values, y_values, "-r", linewidth=2, label="frontera de decision")

plt.title("Frontera de decision del perceptron")
plt.xlabel("Sepal length (cm)")
plt.ylabel("Petal length (cm)")
plt.grid()
plt.legend()
plt.show()

/var/folders/2k/hxbm582j23l33g7ttrfr_wh00000gn/T/ipykernel_50335/2194767117.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


La recta pasa por el hueco entre los dos grupos. Notar que **no** pasa por el medio: el perceptrón se detiene apenas deja de cometer errores, sin buscar el margen más amplio posible. Esa diferencia es justamente lo que distingue al perceptrón de una **máquina de vectores de soporte** (SVM), que sí maximiza el margen.

## 14. Puente con la teoría: la compuerta AND

El mismo estimador, con los 4 puntos de la compuerta AND de las slides (partes 1 a 4), para ver que es el mismo algoritmo aplicado a otro problema.

In [14]:
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

clf_and = Perceptron(max_iter=1000, eta0=0.045, random_state=SEMILLA)
clf_and.fit(X_and, y_and)

print("Esperado: ", y_and)
print("Predicho: ", clf_and.predict(X_and))
print(f"\nPesos: {clf_and.coef_[0]} | sesgo: {clf_and.intercept_[0]}")

Esperado:  [0 0 0 1]
Predicho:  [0 0 0 1]

Pesos: [0.09 0.09] | sesgo: -0.09000000000000001


## 15. El límite: la compuerta XOR

XOR devuelve 1 cuando **exactamente una** de las dos entradas vale 1. Sus cuatro puntos **no son linealmente separables**: no existe ninguna recta que deje `(0,1)` y `(1,0)` de un lado y `(0,0)` y `(1,1)` del otro.

Este es el problema que Minsky y Papert señalaron en 1969 y que frenó la investigación en redes neuronales durante años. La salida no fue abandonar el perceptrón, sino **apilar capas**: un perceptrón multicapa (MLP) sí resuelve el XOR.

In [15]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

clf_xor = Perceptron(max_iter=1000, eta0=0.05, random_state=SEMILLA)
clf_xor.fit(X_xor, y_xor)

print("Esperado: ", y_xor)
print("Predicho: ", clf_xor.predict(X_xor))
print(f"Exactitud: {accuracy_score(y_xor, clf_xor.predict(X_xor)):.2f}  <- nunca llega a 1.00")

Esperado:  [0 1 1 0]
Predicho:  [0 0 0 0]
Exactitud: 0.50  <- nunca llega a 1.00


Por más épocas que se le den, el perceptrón simple **nunca llega al 100%** en XOR: siempre queda al menos un punto mal clasificado. El techo teórico de una recta sobre estos cuatro puntos es 75% (3 de 4), y es habitual que el entrenamiento se estanque antes, como acá, prediciendo todo 0 y quedando en 50%.

## 16. Conclusiones

- El perceptrón resuelve **problemas binarios linealmente separables**; setosa vs versicolor lo es y el modelo acierta todo.
- El modelo entrenado se reduce a los pesos (`coef_`) y el sesgo (`intercept_`), que definen una recta en el plano de atributos.
- La API de scikit-learn (`fit` / `predict` / `score`) encapsula exactamente el algoritmo que en [`practica_redes_neuronales.ipynb`](practica_redes_neuronales.ipynb) se programó a mano.
- Fijar `random_state` no es un detalle: sin eso, cada ejecución da pesos distintos y los resultados no se pueden comparar ni reproducir.
- El XOR marca el límite del modelo de una sola capa y motiva el resto del módulo: funciones de activación, redes multicapa y backpropagation.

## 17. Para seguir practicando

1. Cambiar las clases a `[1, 2]` (versicolor vs virginica), que **no** son linealmente separables. ¿Qué pasa con `n_iter_` y con la exactitud?
2. Bajar `eta0` a `0.001` y subirlo a `1.0`. ¿Cambia la frontera? ¿Cambia la cantidad de épocas?
3. Usar los cuatro atributos en lugar de dos. ¿Mejora la exactitud? (El gráfico ya no se puede dibujar en 2D.)
4. Reemplazar `Perceptron` por `sklearn.neural_network.MLPClassifier` con una capa oculta y volver a probar el XOR de la sección 15.